# Course 4: Mathematics and AI
## Session 6: Category theory for deep learning II: lenses and backprop

**Goal:** write automatic differentiation as composition of lenses, check it against finite differences, and train a small model with it.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. A lens
A lens has two parts:
- `forward(x)` gives the output,
- `backward(x, dy)` takes the input and a gradient with respect to the output, and returns the gradient with respect to the input.

Composition runs forward left to right and backward right to left. That rule is the chain rule, and it is all of backpropagation.

In [ ]:
import numpy as np
class Lens:
    def __init__(self, forward, backward, name="lens"): self.forward, self.backward, self.name = forward, backward, name
    def __rshift__(self, other):
        def fwd(x): return other.forward(self.forward(x))
        def bwd(x, dz):
            y = self.forward(x)                       # recompute the middle value (a real system would cache it)
            return self.backward(x, other.backward(y, dz))
        return Lens(fwd, bwd, f"({self.name} >> {other.name})")

identity = Lens(lambda x: x, lambda x, dy: dy, "id")
relu = Lens(lambda x: np.maximum(x, 0), lambda x, dy: dy * (x > 0), "relu")
tanh = Lens(np.tanh, lambda x, dy: dy * (1 - np.tanh(x) ** 2), "tanh")
def linear(W): return Lens(lambda x: W @ x, lambda x, dy: W.T @ dy, "linear")
def sq_loss(target): return Lens(lambda x: 0.5 * np.sum((x - target) ** 2), lambda x, dy: dy * (x - target), "loss")

### 2. Check against finite differences

In [ ]:
rng = np.random.default_rng(0)
W1, W2, target = rng.normal(size=(5, 3)), rng.normal(size=(2, 5)), rng.normal(size=2)
net = linear(W1) >> tanh >> linear(W2) >> sq_loss(target)
x = rng.normal(size=3)
grad = net.backward(x, 1.0)
numeric = np.array([(net.forward(x + 1e-6 * e) - net.forward(x - 1e-6 * e)) / 2e-6 for e in np.eye(3)])
print("lens gradient   :", grad.round(5)); print("finite difference:", numeric.round(5))
assert np.allclose(grad, numeric, atol=1e-6)

### 3. The laws hold for lenses too
Identity and associativity, tested on both the forward and the backward part. So lenses form a category, and 'take the derivative' is a functor from smooth maps into it: it sends composition to composition. That sentence is the chain rule.

In [ ]:
def same(a, b):
    xs = rng.normal(size=(20, 3)); dy = rng.normal(size=5)
    return all(np.allclose(a.forward(x), b.forward(x)) and np.allclose(a.backward(x, dy), b.backward(x, dy)) for x in xs)
f, g, h = linear(W1), tanh, linear(rng.normal(size=(5, 5)))
print("identity     :", same(identity >> f, f) and same(f >> identity, f))
print("associativity:", same((f >> g) >> h, f >> (g >> h)))
assert same(identity >> f, f) and same((f >> g) >> h, f >> (g >> h))

### 4. Parameters: the Para construction
A layer has two inputs, parameters and data. Treat the pair as the input of one lens. Its backward pass returns two things: the gradient for the parameters (used to learn) and the gradient for the data (passed back to the previous layer). Composing two parametric lenses collects their parameters into a pair.

In [ ]:
class ParaLens:
    """forward(p, x) -> y        backward(p, x, dy) -> (dp, dx)"""
    def __init__(self, forward, backward, init): self.forward, self.backward, self.init = forward, backward, init
    def __rshift__(self, other):
        def fwd(p, x): return other.forward(p[1], self.forward(p[0], x))
        def bwd(p, x, dz):
            y = self.forward(p[0], x); dq, dy = other.backward(p[1], y, dz); dp, dx = self.backward(p[0], x, dy)
            return (dp, dq), dx
        return ParaLens(fwd, bwd, lambda: (self.init(), other.init()))

def dense(m, n, rng):
    return ParaLens(lambda p, x: x @ p[0].T + p[1],
                    lambda p, x, dy: ((dy.T @ x, dy.sum(0)), dy @ p[0]),
                    lambda: (rng.normal(size=(n, m)) * np.sqrt(1 / m), np.zeros(n)))
def activation(fn, dfn): return ParaLens(lambda p, x: fn(x), lambda p, x, dy: ((), dy * dfn(x)), lambda: ())
tanh_p = activation(np.tanh, lambda x: 1 - np.tanh(x) ** 2)

### 5. Train a model with it
An optimizer is also a lens: forward reads the parameter, backward takes the gradient and returns the update. Here it is plain gradient descent, applied to every leaf of the nested parameter tuple.

In [ ]:
def tree_map(fn, *trees):
    if isinstance(trees[0], tuple): return tuple(tree_map(fn, *parts) for parts in zip(*trees))
    return fn(*trees)

rng = np.random.default_rng(0)
X = rng.uniform(-3, 3, size=(256, 1)); Y = np.sin(X)                          # learn the sine function
model = dense(1, 32, rng) >> tanh_p >> dense(32, 1, rng)
params = model.init()
for step in range(3001):
    pred = model.forward(params, X); loss = 0.5 * np.mean((pred - Y) ** 2)
    grads, _ = model.backward(params, X, (pred - Y) / len(X))
    params = tree_map(lambda p, g: p - 0.1 * g, params, grads)
    if step % 1000 == 0: print(f"step {step:4d}   loss {loss:.5f}")
assert loss < 0.01

### 6. Cross-check one gradient with PyTorch

In [ ]:
import torch
p0 = model.init(); (W1_, b1_), _, (W2_, b2_) = p0[0][0], p0[0][1], p0[1]
tW1 = torch.tensor(W1_, requires_grad=True); tb1 = torch.tensor(b1_, requires_grad=True); tW2 = torch.tensor(W2_, requires_grad=True); tb2 = torch.tensor(b2_, requires_grad=True)
tx, ty = torch.tensor(X), torch.tensor(Y)
tl = 0.5 * ((torch.tanh(tx @ tW1.T + tb1) @ tW2.T + tb2 - ty) ** 2).mean(); tl.backward()
pred = model.forward(p0, X); g, _ = model.backward(p0, X, (pred - Y) / len(X))
print("largest difference from PyTorch's gradient for the first weight matrix:", np.abs(g[0][0][0] - tW1.grad.numpy()).max())
assert np.allclose(g[0][0][0], tW1.grad.numpy()) and np.allclose(g[1][0], tW2.grad.numpy())

### TODO
1. Our lens composition recomputes the forward pass inside `backward`. Change the design so the forward pass returns a 'residual' that the backward pass reuses. This is what real autodiff systems store on the tape.
2. Write momentum as a lens with state. What is its 'parameter'?
3. Read section 2 of 'Categorical Foundations of Gradient-Based Learning' (Cruttwell et al.). Match their `Para` and `Lens` to the two classes here, and list one thing the code glosses over.